# Transcribe participant responses

This notebook transcribes each participant's `question_XX.mp4` recordings and builds one combined dataframe with these columns:

1. `participant_id`
2. `question_number`
3. `question_text`
4. `transcript`

The transcription runs locally using faster-whisper. Install the repository requirements before opening the notebook. The first run downloads the selected Whisper model. CSV export will be added later.

## Step 1: Import packages and locate the recordings folder

In [70]:
from pathlib import Path
import re

import pandas as pd
from faster_whisper import WhisperModel

current_directory = Path.cwd().resolve()
repo_root = current_directory.parent
recordings_directory = repo_root / "data" / "recordings"
questions_file = repo_root / "study" / "questions.md"

## Step 2: Specify the participants to transcribe

Edit this list before running the remaining cells. Participant IDs must match the folder names inside `data/recordings/`.

In [71]:
participant_ids = ["P01", "P02", "P03", "P04", "P05", "P06",
                   "P07", "P08", "P09", "P10", "P11", "P12",
                   "P13", "P14", "P15", "P16", "P17", "P18",
                   "P19"]

## Step 3: Load the transcription model

`large-v3` used since it prioritises transcription accuracy.

In [72]:
model_name = "large-v3"
model = WhisperModel(
    model_name,
    device="cpu",
    compute_type="int8",
    use_auth_token=False,
)

## Step 4: Create the questions dataframe

Read `study/questions.md` and extract each question number and its text.

In [73]:
questions_content = questions_file.read_text(encoding="utf-8")
question_headings = list(
    re.finditer(r"^## Question (\d+)\s*$", questions_content, re.MULTILINE)
)
question_rows = []

for index, heading in enumerate(question_headings):
    question_number = int(heading.group(1))
    text_start = heading.end()
    text_end = (
        question_headings[index + 1].start()
        if index + 1 < len(question_headings)
        else len(questions_content)
    )
    question_text = questions_content[text_start:text_end].strip()

    question_rows.append(
        {
            "question_number": question_number,
            "question_text": question_text,
        }
    )

questions_df = pd.DataFrame(
    question_rows,
    columns=["question_number", "question_text"],
)

questions_df

,question_number,question_text
0,0,A bag holds three red marbles and one blue mar...
1,1,A panel of psychologists interviewed 30 nurses...
2,2,Suppose a tennis player reaches the Wimbledon ...
3,3,A cancer test is administered to all the resid...
4,4,Two black and two white marbles are put in an ...
5,5,Two dice are thrown and the product of the two...
6,6,A card is drawn from a well-shuffled standard ...
7,7,60 percent of the population in a city are men...
8,8,A test detects a disease whose prevalence is 1...
9,9,10.3 percent of women in a given city have a p...


## Step 5: Define the transcription helper functions

In [74]:
def get_question_number(recording_path):
    return int(recording_path.stem.split("_")[-1])


def get_participant_recordings(participant_id):
    participant_directory = recordings_directory / participant_id
    session_directories = sorted(participant_directory.glob("session_*"))
    latest_session = session_directories[-1]
    recording_paths = sorted(
        latest_session.glob("question_*.mp4"),
        key=get_question_number,
    )

    if not recording_paths:
        raise FileNotFoundError(f"No question recordings found for {participant_id}.")

    return recording_paths


def transcribe_recording(recording_path):
    segments, _ = model.transcribe(
        str(recording_path),
        language="en",
        beam_size=5,
        vad_filter=True,
    )
    return " ".join(segment.text.strip() for segment in segments).strip()

## Step 6: Load existing transcripts and transcribe new participants

Load the existing CSV and skip any participant whose ID is already present.

In [75]:
columns = ["participant_id", "question_number", "question_text", "transcript"]
output_file = repo_root / "data" / "extracted_data" / "extracted_data.csv"

if output_file.exists():
    transcripts_df = pd.read_csv(output_file)
else:
    transcripts_df = pd.DataFrame(columns=columns)

existing_participants = set(transcripts_df["participant_id"].astype(str))
question_text_by_number = questions_df.set_index("question_number")["question_text"].to_dict()

for participant_id in participant_ids:
    if participant_id in existing_participants:
        print(f"Skipping {participant_id}: participant already transcribed.")
        continue

    participant_rows = []
    recording_paths = get_participant_recordings(participant_id)

    for recording_path in recording_paths:
        question_number = get_question_number(recording_path)

        if question_number == 0:
            print(f"Skipping {participant_id}, question {question_number}.")

        print(f"Transcribing {participant_id}, question {question_number}.")
        participant_rows.append(
            {
                "participant_id": participant_id,
                "question_number": question_number,
                "question_text": question_text_by_number[question_number],
                "transcript": transcribe_recording(recording_path),
            }
        )

    participant_df = pd.DataFrame(participant_rows, columns=columns)
    transcripts_df = pd.concat(
        [transcripts_df, participant_df],
        ignore_index=True,
    )
    existing_participants.add(participant_id)

print("Transcription complete.")

Skipping P01: participant already transcribed.
Skipping P02: participant already transcribed.
Skipping P03: participant already transcribed.
Skipping P04: participant already transcribed.
Skipping P05: participant already transcribed.
Skipping P06: participant already transcribed.
Skipping P07: participant already transcribed.
Skipping P08: participant already transcribed.
Skipping P09: participant already transcribed.
Skipping P10: participant already transcribed.
Skipping P11: participant already transcribed.
Skipping P12: participant already transcribed.
Skipping P13: participant already transcribed.
Skipping P14: participant already transcribed.
Skipping P15: participant already transcribed.
Skipping P16: participant already transcribed.
Skipping P17, question 0.
Transcribing P17, question 0.
Transcribing P17, question 1.
Transcribing P17, question 2.
Transcribing P17, question 3.
Transcribing P17, question 4.
Transcribing P17, question 5.
Transcribing P17, question 6.
Transcribing

## Step 7: Save the combined dataframe

Save all existing and newly created transcripts as `data/extracted_data/extracted_data.csv`.

In [76]:
output_file.parent.mkdir(parents=True, exist_ok=True)
transcripts_df.to_csv(output_file, index=False)

print(f"Transcriptions saved to {output_file}")
transcripts_df

Transcriptions saved to /Users/tandon.utsav2/Library/CloudStorage/OneDrive-SharedLibraries-ImperialCollegeLondon/Salomons, Nicole - PAIR Lab/PAIR Lab- Code Repository/Utsav MSC Codebase/Reachy_app_laptop/data/extracted_data/extracted_data.csv


,participant_id,question_number,question_text,transcript
0,P01,0,A bag holds three red marbles and one blue mar...,"Because you are drawing 1 marble at random, an..."
1,P01,1,A panel of psychologists interviewed 30 nurses...,I decided to use Bayes' Theorem to solve this ...
2,P01,2,Suppose a tennis player reaches the Wimbledon ...,"If the player just wins the first set, then I ..."
3,P01,3,A cancer test is administered to all the resid...,"The second case is more likely than the other,..."
4,P01,4,Two black and two white marbles are put in an ...,the probability that the first marble was whit...
...,...,...,...,...
242,P19,8,A test detects a disease whose prevalence is 1...,"So, the first sentence says, a disease has pre..."
243,P19,9,10.3 percent of women in a given city have a p...,So this question asks us the probability that ...
244,P19,10,Two machines M-1 and M-2 produce balls. Machin...,So we are asked to calculate what is the proba...
245,P19,11,A witness sees a crime involving a taxi in a c...,"So the question is asking us, what is the prob..."
